# Painel Preditivo -- MedData

Explorando previsao de curto prazo (internacoes diarias) usando suavizacao exponencial
(Holt-Winters / ETS), a partir dos mesmos dados que alimentam a Visao Geral e o
Monitoramento (via `data_loader.py`).

**Rode este notebook com o kernel do `.venv` do projeto** (Python 3.12) -- o mesmo
usado pelo `streamlit run app.py`. Python 3.14 (o Python "de sistema" nesta maquina) tem
um bug conhecido de segfault com pandas/datetime64 -- ver README.md.

## O que este notebook faz (e o que nao faz)

- **Faz**: mostra que da pra prever tendencia de curto prazo (proximas 1-4 semanas) e
  captar o padrao **semanal** (dia da semana), porque isso se repete ~52 vezes em 12
  meses de dado diario -- estatisticamente solido.
- **NAO faz**: nao tenta capturar sazonalidade **anual** (ex.: "internacoes sobem em
  julho todo ano") -- os dados tem so ~12 meses de volume bom (ver nota de qualidade de
  dado abaixo), entao um padrao anual so foi visto acontecer uma vez -- nao da pra
  confirmar que e um padrao de verdade e nao coincidencia.
- Isso e uma exploracao/prototipo (Nivel 2 da conversa sobre "painel preditivo") --
  antes de virar uma pagina do Streamlit, vale rodar este notebook com mais tempo/dado
  conforme o projeto avancar.


In [ ]:
import sys
from pathlib import Path

# Caminho do projeto (ajuste se mover este notebook de lugar)
PROJETO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJETO_DIR))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

from core.data_loader import load_all
from core.tema_editorial import aplicar_tema_claro

pd.options.display.float_format = "{:.2f}".format


## 1. Carregar os dados

Reaproveita `data_loader.load_all()` -- os mesmos dados (V1 ou V2, conforme
`VERSAO_ATIVA` em `data_loader.py`) usados pela Visao Geral. So precisamos da
`data_internacao` do fato para montar a serie diaria.


In [2]:
dados = load_all()
fato = dados["fato_internacao"]

print(f"Linhas em fato_internacao: {len(fato):,}".replace(",", "."))
print(f"Periodo coberto: {fato['data_internacao'].min().date()} a {fato['data_internacao'].max().date()}")
fato[["data_internacao"]].head()


2026-09-11 07:11:34.162 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-11 07:11:34.165 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-09-11 07:11:35.275 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-11 07:11:35.277 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-09-11 07:11:35.293 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


...

2026-09-11 07:11:35.348 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-09-11 07:11:35.450 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-11 07:11:35.452 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Linhas em fato_internacao: 692.929
Periodo coberto: 2008-01-01 a 2024-11-30


,data_internacao
0,2024-01-15
1,2024-01-10
2,2024-01-23
3,2023-12-25
4,2023-12-28


## 2. Montar a serie diaria de internacoes

Conta quantas internacoes comecaram em cada dia. Usa `reindex` pra preencher com 0 os
dias sem nenhuma internacao registrada (senao eles simplesmente "sumiriam" da serie em
vez de aparecer como zero).


In [3]:
serie_diaria = (
    fato.groupby(fato["data_internacao"].dt.floor("D")).size()
)
indice_completo = pd.date_range(serie_diaria.index.min(), serie_diaria.index.max(), freq="D")
serie_diaria = serie_diaria.reindex(indice_completo, fill_value=0)
serie_diaria.index.name = "data"
serie_diaria.name = "internacoes"

print(f"Serie diaria: {len(serie_diaria)} dias, de {serie_diaria.index.min().date()} a {serie_diaria.index.max().date()}")
serie_diaria.describe()


Serie diaria: 6179 dias, de 2008-01-01 a 2024-11-30


count   6179.00
mean     112.14
std      485.02
min        0.00
25%        0.00
50%        0.00
75%        0.00
max     3351.00
Name: internacoes, dtype: float64

In [ ]:
fig = go.Figure(go.Scatter(x=serie_diaria.index, y=serie_diaria.values, mode="lines", name="Internacoes/dia"))
fig.update_layout(title="Internacoes por dia -- serie completa", xaxis_title="Data", yaxis_title="Internacoes")
fig = aplicar_tema_claro(fig, altura=350)
fig.show()


## 3. Existe um padrao semanal?

Antes de montar qualquer modelo, vale checar visualmente se o dia da semana importa --
se nao importar, nao faz sentido pedir pro modelo capturar sazonalidade semanal.


In [ ]:
dias_semana_pt = ["Segunda", "Terca", "Quarta", "Quinta", "Sexta", "Sabado", "Domingo"]
por_dia_semana = serie_diaria.groupby(serie_diaria.index.dayofweek)

fig = go.Figure()
for dia_num, grupo in por_dia_semana:
    fig.add_trace(go.Box(y=grupo.values, name=dias_semana_pt[dia_num]))
fig.update_layout(title="Internacoes por dia da semana (distribuicao)", yaxis_title="Internacoes/dia", showlegend=False)
fig = aplicar_tema_claro(fig, altura=350)
fig.show()


## 4. Recorte do dado "bom"

Conforme ja documentado no projeto (ver `rag.py`), o volume real de internacoes esta
concentrado nos ultimos ~12 meses de dados -- registros mais antigos podem ser
backlog/dado incompleto.

**Achado novo (descoberto rodando este notebook)**: os ultimos dias da serie tem uma
queda artificial -- os dados parecem ter sido extraidos no meio do processamento do
ultimo mes, entao os dias mais recentes aparecem sub-contados (nao e uma queda real de
internacoes). Ver a celula abaixo.


In [6]:
print("Ultimos 14 dias da serie diaria (antes de qualquer corte):")
print(serie_diaria.tail(14).to_string())


Ultimos 14 dias da serie diaria (antes de qualquer corte):
data
2024-11-17    1306
2024-11-18    2044
2024-11-19    1820
2024-11-20    1451
2024-11-21    1755
2024-11-22    1414
2024-11-23     979
2024-11-24     866
2024-11-25    1399
2024-11-26    1228
2024-11-27    1031
2024-11-28     739
2024-11-29     326
2024-11-30      63
Freq: D


Os ultimos dias caem de forma abrupta e implausivel (de ~1.000+/dia pra 63 no ultimo
dia) -- um sinal classico de corte de extracao no meio do periodo, nao uma tendencia
real. Se isso entrar no treino do modelo sem tratamento, ele confunde com "tendencia de
queda" e extrapola pra valores negativos (foi exatamente o que aconteceu na primeira
versao deste notebook -- ver nota no final). Por isso, cortamos essa borda antes de
definir a janela usada no forecast.

**Isso tambem e relevante pro resto do dashboard**: qualquer grafico que mostre o
ultimo mes disponivel "cru" (ex.: a serie mensal da Visao Geral) pode estar mostrando
um ultimo ponto artificialmente baixo por esse mesmo motivo -- vale documentar isso
como limitacao conhecida de qualidade de dado.


In [7]:
DIAS_BORDA_NAO_CONFIAVEL = 14  # margem de seguranca; os primeiros ~5-7 dias ja mostram queda clara

fim_dado_bom = serie_diaria.index.max() - pd.Timedelta(days=DIAS_BORDA_NAO_CONFIAVEL)
inicio_dado_bom = fim_dado_bom - pd.Timedelta(days=364)  # ~12 meses, terminando antes da borda
serie_recente = serie_diaria.loc[inicio_dado_bom:fim_dado_bom].asfreq("D")

print(f"Janela usada no forecast: {serie_recente.index.min().date()} a {serie_recente.index.max().date()} ({len(serie_recente)} dias)")
print(f"(Os ultimos {DIAS_BORDA_NAO_CONFIAVEL} dias da base, ate {serie_diaria.index.max().date()}, foram excluidos por serem borda nao confiavel.)")
serie_recente.describe()


Janela usada no forecast: 2023-11-18 a 2024-11-16 (365 dias)
(Os ultimos 14 dias da base, ate 2024-11-30, foram excluidos por serem borda nao confiavel.)


count    365.00
mean    1823.89
std      895.77
min      202.00
25%      977.00
50%     1922.00
75%     2625.00
max     3351.00
Name: internacoes, dtype: float64

## 5. Validacao: treino/teste

Antes de confiar em qualquer previsao, escondemos os ultimos 21 dias reais, treinamos
so com o resto, e comparamos previsto vs. real nesse periodo escondido. Isso da um erro
medio honesto (MAPE) em vez de so acreditar no modelo.

Usamos `damped_trend=True` (tendencia amortecida) como protecao extra: em vez de deixar
a tendencia continuar pra sempre em linha reta, ela vai perdendo forca conforme o
horizonte de previsao aumenta -- evita o tipo de extrapolacao catastrofica (valores
negativos, sem sentido) que aconteceu numa primeira tentativa deste notebook, antes de
identificarmos e cortar a borda de dado nao confiavel na secao anterior.


In [8]:
HORIZONTE_VALIDACAO = 21  # dias

treino = serie_recente.iloc[:-HORIZONTE_VALIDACAO]
teste = serie_recente.iloc[-HORIZONTE_VALIDACAO:]

modelo_validacao = ETSModel(
    treino, error="add", trend="add", damped_trend=True, seasonal="add", seasonal_periods=7
)
ajuste_validacao = modelo_validacao.fit(disp=False)

previsao_teste = ajuste_validacao.get_prediction(start=teste.index[0], end=teste.index[-1])
resumo_teste = previsao_teste.summary_frame(alpha=0.05)

mape = float(np.mean(np.abs((teste.values - resumo_teste["mean"].values) / np.maximum(teste.values, 1))) * 100)
rmse = float(np.sqrt(np.mean((teste.values - resumo_teste["mean"].values) ** 2)))
print(f"MAPE na janela de validacao (ultimos {HORIZONTE_VALIDACAO} dias reais): {mape:.1f}%")
print(f"RMSE: {rmse:.1f} internacoes/dia")


MAPE na janela de validacao (ultimos 21 dias reais): 22.4%
RMSE: 513.3 internacoes/dia


In [ ]:
# pi_lower travado em 0 -- nao existe internacao negativa, e o modelo (erro aditivo)
# pode produzir um limite inferior levemente negativo quando a incerteza e grande.
limite_inferior = resumo_teste["pi_lower"].clip(lower=0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=teste.index, y=teste.values, mode="lines+markers", name="Real"))
fig.add_trace(go.Scatter(x=resumo_teste.index, y=resumo_teste["mean"], mode="lines", name="Previsto"))
fig.add_trace(go.Scatter(
    x=list(resumo_teste.index) + list(resumo_teste.index[::-1]),
    y=list(resumo_teste["pi_upper"]) + list(limite_inferior[::-1]),
    fill="toself", fillcolor="rgba(45,212,191,0.15)", line=dict(color="rgba(0,0,0,0)"),
    name="Intervalo de confianca (95%)", showlegend=True,
))
fig.update_layout(title=f"Validacao: previsto vs. real (ultimos {HORIZONTE_VALIDACAO} dias)", xaxis_title="Data", yaxis_title="Internacoes")
fig = aplicar_tema_claro(fig, altura=350)
fig.show()


**Como ler o MAPE**: e o erro percentual medio entre o previsto e o real. Um MAPE de,
por exemplo, 15% significa que a previsao erra em media 15% pra mais ou pra menos --
util pra decidir se vale apresentar isso como "previsao confiavel" ou como algo que
ainda precisa de mais trabalho antes de ir pro dashboard principal.


## 6. Previsao final: proximos dias

Com a validacao feita (e o erro conhecido), retreinamos com TODO o dado recente
disponivel (sem esconder nada) e projetamos os proximos dias de verdade.


In [10]:
HORIZONTE_PREVISAO = 21  # dias a frente

modelo_final = ETSModel(
    serie_recente, error="add", trend="add", damped_trend=True, seasonal="add", seasonal_periods=7
)
ajuste_final = modelo_final.fit(disp=False)

datas_futuras = pd.date_range(serie_recente.index.max() + pd.Timedelta(days=1), periods=HORIZONTE_PREVISAO, freq="D")
previsao_final = ajuste_final.get_prediction(start=datas_futuras[0], end=datas_futuras[-1])
resumo_final = previsao_final.summary_frame(alpha=0.05)

resumo_final[["mean", "pi_lower", "pi_upper"]].round(1)


,mean,pi_lower,pi_upper
2024-11-17,1058.90,640.90,1476.90
2024-11-18,1840.70,1369.70,2311.80
2024-11-19,1789.00,1263.00,2315.00
2024-11-20,1807.90,1225.80,2390.10
2024-11-21,1718.00,1079.00,2356.90
2024-11-22,1391.00,695.00,2087.00
2024-11-23,1083.50,330.60,1836.50
2024-11-24,929.10,15.60,1842.50
2024-11-25,1723.20,759.60,2686.90
2024-11-26,1682.70,668.70,2696.60


In [ ]:
ultimos_90_dias = serie_recente.iloc[-90:]
limite_inferior_final = resumo_final["pi_lower"].clip(lower=0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ultimos_90_dias.index, y=ultimos_90_dias.values, mode="lines", name="Historico (real)"))
fig.add_trace(go.Scatter(x=resumo_final.index, y=resumo_final["mean"], mode="lines", line=dict(dash="dash"), name="Previsao"))
fig.add_trace(go.Scatter(
    x=list(resumo_final.index) + list(resumo_final.index[::-1]),
    y=list(resumo_final["pi_upper"]) + list(limite_inferior_final[::-1]),
    fill="toself", fillcolor="rgba(45,212,191,0.15)", line=dict(color="rgba(0,0,0,0)"),
    name="Intervalo de confianca (95%)", showlegend=True,
))
fig.update_layout(
    title=f"Previsao para os proximos {HORIZONTE_PREVISAO} dias",
    xaxis_title="Data", yaxis_title="Internacoes/dia",
)
fig = aplicar_tema_claro(fig, altura=400)
fig.show()


## Conclusoes e proximos passos

- **Achado mais importante deste notebook**: os ultimos ~10-14 dias disponiveis na base
  tem contagem artificialmente baixa (borda de extracao dos dados, nao uma queda real
  de internacoes). Isso quebrava o forecast por completo antes de identificarmos e
  cortar essa borda (a primeira tentativa gerou previsao negativa, sem sentido). Vale
  levar esse achado pro resto do projeto -- qualquer grafico que mostre o ultimo ponto
  "cru" da serie temporal (ex.: o ultimo mes na Visao Geral) pode estar mostrando um
  numero artificialmente baixo pelo mesmo motivo.
- Este notebook mostra um forecast de **curto prazo** (semanas, nao meses/anos) usando
  so tendencia (amortecida) + padrao semanal -- nada de sazonalidade anual, que os
  dados nao sustentam com confianca ainda (ver secao 3).
- O **MAPE da validacao** (secao 5) e o numero mais importante pra decidir se isso ja
  esta bom o suficiente pra virar uma pagina do dashboard, ou se precisa de mais
  trabalho (ex.: mais dado, ajustar o horizonte de previsao pra mais curto, tratar
  outliers).
- **Antes de expor isso num painel pro usuario final**: rodar essa validacao de novo
  quando houver mais dados, e deixar claro na interface que e uma projecao estatistica
  de tendencia, nao uma certeza -- igual discutimos, mostrar o intervalo de confianca
  (nao so um numero seco) e essencial pra nao vender uma precisao que o modelo nao tem.
- **Proximo passo natural**: se o MAPE for aceitavel, portar a logica das secoes 4 e 6
  pra uma funcao em `consultas.py` (`prever_internacoes_diarias(...)`) e criar uma nova
  secao/pagina que chama essa funcao e plota o mesmo grafico dentro do app Streamlit.
